---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-33: LangChain <b>Chains</b> Component</h1>

# Learning agenda of this notebook

> **The idea of this lecture:** real tasks need more than one step: fill a prompt, call a model, clean the answer, then maybe call another model with that answer. A **chain** joins these steps with the `|` operator, so that the output of one step becomes the input of the next, and the whole pipeline runs with a single `invoke()`. This way of writing chains is called **LCEL**, the LangChain Expression Language.

1. **Where Chains Fit**
    - What a chain is, and what happened to the old chain classes
    - Setup: check the API keys
2. **Your First Chain**
    - Example 1: `prompt | model | parser`, compared with the same three steps by hand
    - Example 2: A chain is a Runnable too: `batch()` and `stream()`
3. **Output Parsers**
    - Example 3: `PydanticOutputParser`: from the model's text to a Python object
4. **Three Shapes of Chains**
    - Example 4: Sequential: translate, then summarise
    - Example 5: Parallel: two reports at the same time, then one merged report
    - Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback
5. **Making Chains Robust**
    - Example 7: `with_retry()` and `with_fallbacks()`
6. **Where Chains Go Next**
7. **Summary**

> **Note:** all examples use free models: `openai/gpt-oss-120b` on Groq, and a free OpenRouter model as the backup in Example 7.

# <span style='background :lightgreen' >1. Where Chains Fit</span>

In the app of Lec-30, a **chain** is what connects the building blocks: retriever, prompt template and model, one after the other.

<div style="text-align:center;">
    <img src="../images/langchain-new-03.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## What is a chain?
- In Lec-31 we filled a template with `invoke()` and gave the result to a model with another `invoke()`. Prompt templates, models and output parsers all have the **same** `invoke()` method: in LangChain they are all **Runnables**.
- Because they share one interface, they can be joined with `|`: `prompt | model | parser` is a new Runnable that calls the three in order and passes each output to the next. This is a **chain**.
- A chain is a Runnable again, so it has the same methods (`invoke`, `batch`, `stream`, …) and can itself be a step of a bigger chain.

## What happened to the old chain classes
Older tutorials build chains with classes. They still work, but they were moved to `langchain-classic` in LangChain 1.0 and print a deprecation warning:

| Old class (`langchain_classic.chains`) | Today, with LCEL (`langchain_core`) |
|---|---|
| `LLMChain(llm=model, prompt=prompt)` | `prompt \| model \| StrOutputParser()` (Example 1) |
| `SimpleSequentialChain`, `SequentialChain` | `chain_1 \| reshape \| chain_2` (Example 4) |
| `RouterChain`, `MultiPromptChain` | `RunnableBranch(...)` (Example 6) |

For workflows with **loops**, saved **state** or several **agents**, LangChain now recommends **LangGraph**, which we meet after the agent lectures. Chains are for fixed pipelines that run from start to end.

> **Documentation:** https://reference.langchain.com/python/langchain-core/runnables

## Setup: check the API keys
- Every example below is **standalone**: its first code cell loads the API keys and creates the model with `init_chat_model("groq:openai/gpt-oss-120b")` (Lec-30). The later cells of the same example build on it, so run them in order.
- To use OpenAI instead, uncomment the `"openai:gpt-4o-mini"` line in an example's first cell and comment out the Groq line above it.
- This cell only checks that the keys are found; no example depends on it.

In [1]:
import os                                                   # read the environment variables
from dotenv import load_dotenv                              # load the API keys from ../keys/.env

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ (the file wins)

for name in ["GROQ_API_KEY", "OPENROUTER_API_KEY"]:        # Groq in every example; OpenRouter for the backup in Example 7
    if os.getenv(name):
        print(f"{name:18} set")
    else:
        print(f"{name:18} MISSING")                         # never print any part of a key

GROQ_API_KEY       set
OPENROUTER_API_KEY set


# <span style='background :lightgreen' >2. Your First Chain</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-16.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 1: <code>prompt | model | parser</code>, compared with the same three steps by hand</span>
### Step 1: Three steps by hand
We translate a short biography into Urdu. Each step's output is passed **by our code** to the next step.

In [2]:
# Step 1a: load the API keys, create the model, and write the prompt template
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

translate_prompt = PromptTemplate.from_template(
    "Translate the following English text into {language}. Reply with the translation only.\n\n{text}"
)

In [3]:
# Step 1b: fill the template → a PromptValue
values = {"language": "Urdu", "text": BIO}                  # one value for each {placeholder}
prompt_value = translate_prompt.invoke(values)
print(type(prompt_value).__name__)                          # StringPromptValue
print(prompt_value.to_string()[:150], "…")                  # the filled prompt (first 150 characters)

StringPromptValue
Translate the following English text into Urdu. Reply with the translation only.

Dr. Muhammad Arif Butt is an Assistant Professor at the Department o …


In [4]:
# Step 1c: call the model → an AIMessage
ai_message = model.invoke(prompt_value)
print(type(ai_message).__name__)                            # AIMessage

AIMessage


In [5]:
# Step 1d: keep only the text → a str
translation = ai_message.text
print(translation)

ڈاکٹر محمد عارف بٹ ڈیپارٹمنٹ آف ڈیٹا سائنس، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ انہوں نے ایم ایس سی، ایم فل (دونوں گولڈ میڈل کے ساتھ) اور کمپیوٹر سائنس میں پی ایچ ڈی حاصل کی ہے۔ ۳۳ سال سے زیادہ تدریسی اور انتظامی تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور مصنوعی ذہانت پڑھاتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیالیئٹ اور فیلکن ہنٹ کے بانی ہیں۔


### Step 2: The same three steps as one chain
- `StrOutputParser()` is a small component that takes an `AIMessage` and returns only its text, the same as `.text` in Step 1. As a component, it can be the last step of a chain.
- `translate_prompt | model | parser` builds the chain; one `invoke()` runs all three steps.
- `chain.get_graph().print_ascii()` draws the chain: the input, each component, and the output.

In [6]:
# Step 2a: the same three steps as ONE chain
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text (a str), like .text in Step 1d

parser = StrOutputParser()
translate_chain = translate_prompt | model | parser         # the output of each step goes into the next

In [7]:
# Step 2b: one invoke() runs all three steps
result = translate_chain.invoke(values)                     # the same input dict as Step 1b
print("Is the result a str?", isinstance(result, str), "\n")   # yes: the parser's output is the chain's output
print(result)

Is the result a str? True 

ڈاکٹر محمد عارف بٹ ڈیپارٹمنٹ آف ڈیٹا سائنس، یونیورسٹی آف دی پنجاب، لاہور میں اسسٹنٹ پروفیسر ہیں۔ اُن کے پاس ایم ایس سی، ایم فل (دونوں کے ساتھ گولڈ میڈل) اور کمپیوٹر سائنس میں پی ایچ ڈی ہے۔ 33 سال سے زائد تدریس اور انتظامیہ کے تجربے کے ساتھ، وہ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سکیورٹی اور مصنوعی ذہانت کی تدریس کرتے ہیں۔ وہ ایک ٹیکنالوجی انٹرپرینیور بھی ہیں اور ایکسیلیئٹ اور فیلکن-ہنٹ کے بانی ہیں۔


In [8]:
# Step 2c: a drawing of the chain: the input, each component, and the output
graph = translate_chain.get_graph()
graph.print_ascii()                                         # needs the grandalf package

     +-------------+       
     | PromptInput |       
     +-------------+       
            *              
            *              
            *              
    +----------------+     
    | PromptTemplate |     
    +----------------+     
            *              
            *              
            *              
      +----------+         
      | ChatGroq |         
      +----------+         
            *              
            *              
            *              
   +-----------------+     
   | StrOutputParser |     
   +-----------------+     
            *              
            *              
            *              
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


## <span style='color:#7e22ce'>Example 2: A chain is a Runnable too: <code>batch()</code> and <code>stream()</code></span>
The methods of a model (Lec-30, Example 8) work on a whole chain:
- `batch([...])` runs the chain on several inputs at the same time and returns the outputs in the same order.
- `stream(...)` yields the output piece by piece. Because the last step is `StrOutputParser`, the pieces are plain strings.

In [9]:
# Step a: load the API keys, create the model, and build the translate chain of Example 1
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text (a str)

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

translate_prompt = PromptTemplate.from_template(
    "Translate the following English text into {language}. Reply with the translation only.\n\n{text}"
)
translate_chain = translate_prompt | model | StrOutputParser()

In [10]:
# Step b: batch(): one sentence into three languages, all at the same time
sentence = "Knowledge is the light that guides us through the darkest times."
languages = ["Urdu", "French", "Arabic"]

inputs = []                                                 # one input dict per language
for language in languages:
    inputs.append({"language": language, "text": sentence})

outputs = translate_chain.batch(inputs)                     # a list of str, in the same order as inputs
for index in range(len(languages)):
    print(f"{languages[index]:7} {outputs[index]}")

Urdu    علم وہ روشنی ہے جو ہمیں سب سے تاریک اوقات میں رہنمائی کرتی ہے۔
French  La connaissance est la lumière qui nous guide à travers les temps les plus sombres.
Arabic  المعرفة هي النور الذي يوجهنا خلال أحلك الأوقات.


In [11]:
# Step c: stream(): the answer piece by piece, as soon as each piece comes
values = {"language": "Spanish", "text": BIO}
for piece in translate_chain.stream(values):                # each piece is a str (thanks to StrOutputParser)
    print(piece, end="", flush=True)                        # end="": no new line; flush=True: show it at once

El Dr. Muhammad Arif Butt es Profesor Asistente en el Departamento de Ciencia de Datos de la Universidad del Punjab, Lahore. Posee una Maestría, un MPhil (ambos con medallas de oro) y un doctorado en Ciencias de la Computación. Con más de 33 años de experiencia en enseñanza y gestión, imparte cursos de sistemas operativos, programación de sistemas, ciberseguridad e inteligencia artificial. Además, es emprendedor tecnológico y fundador de Excaliat y Falcon‑Hunt.

# <span style='background :lightgreen' >3. Output Parsers</span>

An **output parser** is the last step of many chains: it turns the model's answer into the type our program needs.

| Parser | Turns the answer into | How |
|---|---|---|
| `StrOutputParser()` | a `str` | takes the text of the `AIMessage` |
| `PydanticOutputParser(pydantic_object=Movie)` | a validated `Movie` object | asks for JSON in the prompt, then reads and checks it |

## <span style='color:#7e22ce'>Example 3: <code>PydanticOutputParser</code>: from the model's text to a Python object</span>
It works in two halves:
1. **Before the model:** `parser.get_format_instructions()` writes instructions that describe the JSON we want, from our Pydantic class. We put them into the prompt with `partial()` (Lec-31, Example 3).
2. **After the model:** the parser reads the JSON in the model's text, checks every field and its type, and returns a `Movie` object. If the JSON is wrong, it raises an `OutputParserException` instead of letting bad data into the program.

In [12]:
# Step a: load the API keys, create the model, describe the answer we want, and make its parser
from datetime import date                                   # today's date, to compute the film's age
from pydantic import BaseModel, Field                       # describe (and validate) the shape of the answer
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)
from langchain_core.output_parsers import PydanticOutputParser   # the model's JSON text → a Pydantic object

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

class Movie(BaseModel):                                     # the object we want back
    title: str = Field(description="the film's title")
    year: int = Field(description="the year the film was released")
    director: str = Field(description="the director's full name")

movie_parser = PydanticOutputParser(pydantic_object=Movie)

# Half 1, BEFORE the model: instructions that describe the JSON we want
format_instructions = movie_parser.get_format_instructions()
print(format_instructions)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"title": {"description": "the film's title", "title": "Title", "type": "string"}, "year": {"description": "the year the film was released", "title": "Year", "type": "integer"}, "director": {"description": "the director's full name", "title": "Director", "type": "string"}}, "required": ["title", "year", "director"]}
```


In [13]:
# Step b: put the format instructions into the prompt with partial()
template = PromptTemplate.from_template("Answer the question.\n{format_instructions}\nQuestion: {question}")
movie_prompt = template.partial(format_instructions=format_instructions)   # fill this slot now, {question} later
print("Still to fill:", movie_prompt.input_variables)      # only ['question'] is left

question = {"question": "Which film did Francis Ford Coppola release in 1972 about a mafia family?"}
filled = movie_prompt.invoke(question)
print("\nThe end of the prompt the model will see:\n…", filled.to_string()[-250:])

Still to fill: ['question']

The end of the prompt the model will see:
… ", "type": "integer"}, "director": {"description": "the director's full name", "title": "Director", "type": "string"}}, "required": ["title", "year", "director"]}
```
Question: Which film did Francis Ford Coppola release in 1972 about a mafia family?


In [14]:
# Step c: Half 2, AFTER the model: the parser reads the JSON and returns a Movie object
movie_chain = movie_prompt | model | movie_parser           # the parser is the last step

movie = movie_chain.invoke(question)
print(type(movie).__name__, "→", movie)                     # a Movie object, not a string
age = date.today().year - movie.year                        # year is an int, so we can compute with it
print(f"{movie.title} is {age} years old.")

Movie → title='The Godfather' year=1972 director='Francis Ford Coppola'
The Godfather is 54 years old.


> **`PydanticOutputParser` or `with_structured_output()`?** Both give a validated object. `with_structured_output()` (Lec-30, Example 9) uses the model provider's built-in JSON mode or tool calling, so it is more reliable when the model supports it. `PydanticOutputParser` only needs text, so it works with **any** model, including small local ones, and many RAG tutorials use it.

# <span style='background :lightgreen' >4. Three Shapes of Chains</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-17.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 4: Sequential: translate, then summarise</span>
- We join two chains: `translate_chain` (built again as in Example 1) and a new `summarize_chain`. But they do not fit together directly: `translate_chain` returns a **str**, and `summarize_prompt` needs a **dict** with `language` and `text`.
- **`RunnableLambda(function)`** turns any Python function into a Runnable, so it can sit between them with `|`. Here it reshapes the Urdu string into the dict the next prompt expects.
- (A plain function or lambda next to `|` is turned into a `RunnableLambda` automatically; writing it out makes the step easy to see.)

In [15]:
# Step a: load the API keys, create the model, and build the two chains
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text
from langchain_core.runnables import RunnableLambda         # turns a Python function into a chain step

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

BIO = '''Dr. Muhammad Arif Butt is an Assistant Professor at the Department of Data Science, University of the Punjab, Lahore.
He holds an MSc, an MPhil (both with gold medals) and a Ph.D. in Computer Science. With over 33 years of experience in teaching
and management, he teaches operating systems, system programming, cybersecurity and artificial intelligence. He is also a
technology entrepreneur and the founder of Excaliat and Falcon-Hunt.'''

# Chain 1: translate (as in Example 1)
translate_prompt = PromptTemplate.from_template(
    "Translate the following English text into {language}. Reply with the translation only.\n\n{text}"
)
translate_chain = translate_prompt | model | StrOutputParser()

# Chain 2: summarise
summarize_prompt = PromptTemplate.from_template(
    "Summarise the following {language} text in fewer than 50 words, in {language}:\n\n{text}"
)
summarize_chain = summarize_prompt | model | StrOutputParser()

In [16]:
# Step b: the two chains do not fit: translate_chain returns a str, summarize_prompt needs a dict
print("summarize_prompt needs:", summarize_prompt.input_variables)

def make_summary_input(urdu_text):
    '''str → the dict that summarize_prompt needs.'''
    return {"language": "Urdu", "text": urdu_text}

to_summary_input = RunnableLambda(make_summary_input)       # the function as a chain step

print("A test, without any model:", to_summary_input.invoke("یہ ایک مثال ہے۔"))

summarize_prompt needs: ['language', 'text']
A test, without any model: {'language': 'Urdu', 'text': 'یہ ایک مثال ہے۔'}


In [17]:
# Step c: join the three steps into one chain, and run it once
translate_then_summarize = translate_chain | to_summary_input | summarize_chain

values = {"language": "Urdu", "text": BIO}
summary = translate_then_summarize.invoke(values)           # English → Urdu → a short Urdu summary
print(summary)

ڈاکٹر محمد عارف بٹ، یونیورسٹی آف دی پنجاب، لاہور کے اسسٹنٹ پروفیسر، ایم ایس سی، ایم فل (گولڈ میڈل) اور پی ایچ ڈی کے حامل ہیں۔ 33 سال کے تجربے کے ساتھ آپریٹنگ سسٹمز، سسٹم پروگرامنگ، سائبر سیکیورٹی اور AI پڑھاتے ہیں اور ایکسیلیئٹ و فیلکن‑ہنٹ کے بانی ہیں۔


## <span style='color:#7e22ce'>Example 5: Parallel: two reports at the same time, then one merged report</span>
- **`RunnableParallel({"report_a": chain_a, "report_b": chain_b})`** gives the **same** input to both chains, runs them **at the same time**, and returns one dict with both results under the keys we chose.
- That dict is exactly what the next prompt needs: `{report_a}` and `{report_b}`, so `| merge_chain` follows directly.
- `partial()` (Lec-31) makes two versions of one report template, each with its own perspective fixed, so each needs only the `topic`.
- (A plain dict `{"report_a": chain_a, "report_b": chain_b}` next to `|` means the same as `RunnableParallel(...)`.)
- Running at the same time saves time when the provider serves several requests at once. On Groq's free tier, the limit of about 8,000 tokens per minute can make the second report wait for the first, so the gain varies from minute to minute; with a paid account or a local model, two reports take about as long as one.

In [18]:
# Step a: load the API keys, create the model, and build one report chain per perspective
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text
from langchain_core.runnables import RunnableParallel       # runs several chains on the same input, at the same time

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

# "in about 300 words" keeps both reports short enough for the merge step: Groq's free tier rejects
# any single request above 8,000 tokens (HTTP 413), and two long reports together can exceed that
report_prompt = PromptTemplate.from_template(
    "Write a detailed analytical report, in about 300 words, about the {topic} from {perspective}."
)

american_prompt = report_prompt.partial(perspective="the American perspective")              # only {topic} is left
chain_american = american_prompt | model | StrOutputParser()

middle_eastern_prompt = report_prompt.partial(perspective="the Middle Eastern perspective")  # only {topic} is left
chain_middle_eastern = middle_eastern_prompt | model | StrOutputParser()

In [19]:
# Step b: RunnableParallel gives the SAME input to both chains, and returns one dict
both_reports = RunnableParallel({
    "report_a": chain_american,                             # key → the chain that fills it
    "report_b": chain_middle_eastern,
})

topic = {"topic": "9/11 incident"}
reports = both_reports.invoke(topic)                        # both chains run at the same time
print(type(reports).__name__, "with the keys", list(reports.keys()), "\n")
for key in reports:
    print(f"{key}: {reports[key][:200]} …\n")               # the first 200 characters of each report

dict with the keys ['report_a', 'report_b'] 

report_a: **Analytical Report – The September 11, 2001 Terrorist Attacks (American Perspective)**  

**Executive Summary**  
On 11 September 2001, four coordinated hijackings by al‑Qaeda operatives resulted in  …

report_b: **Analytical Report: The September 11 2001 Attacks – A Middle‑Eastern Perspective (≈300 words)**  

The September 11 attacks on the United States have been interpreted across the Middle East through a …



In [20]:
# Step c: that dict is exactly what the merge prompt needs: {report_a} and {report_b}
merge_prompt = PromptTemplate.from_template(
    "Combine the following two reports into one unified report of fewer than 50 words.\n"
    "Report A: {report_a}\nReport B: {report_b}"
)
merge_chain = merge_prompt | model | StrOutputParser()

merged = merge_chain.invoke(reports)                        # the dict of Step b goes straight in
print(merged)

9/11 killed 2,977, spurred U.S. Patriot Act, DHS, wars in Afghanistan/Iraq, and a security‑focused culture; regionally, it reinforced narratives of U.S. imperialism, sparked divergent religious reactions, justified authoritarian crackdowns, fueled anti‑Muslim backlash, and deepened cycles of violence that still shape geopolitics.


In [21]:
# Step d: the whole pipeline as one chain: {topic} → {report_a, report_b} → one report
final_chain = both_reports | merge_chain

print(final_chain.invoke(topic), "\n")                      # two reports at the same time, then the merge
graph = final_chain.get_graph()
graph.print_ascii()

On 9/11 al‑Qaeda hijacked four planes, killing 2,977, prompting U.S. emergency response, the PATRIOT Act, DHS, and wars in Afghanistan and Iraq. Middle‑Eastern governments largely condemned the attacks; public opinion mixed sympathy with anti‑imperialism, and media framed them as a symptom of U.S. policy, reshaping regional politics. 

       +----------------------------------+        
       | Parallel<report_a,report_b>Input |        
       +----------------------------------+        
                ***             ***                
              **                   **              
            **                       **            
+----------------+              +----------------+ 
| PromptTemplate |              | PromptTemplate | 
+----------------+              +----------------+ 
          *                             *          
          *                             *          
          *                             *          
    +----------+                  +---

## <span style='color:#7e22ce'>Example 6: Branching (if-else): a different reply for positive, negative and mixed feedback</span>
- **Step 1, classify and keep:** a classifier gives the feedback's sentiment. **`RunnablePassthrough.assign(sentiment=classify)`** passes the input dict **through unchanged** and **adds** a new key with the classifier's result: `{"feedback": …}` becomes `{"feedback": …, "sentiment": "mixed"}`. Without `.assign()`, the next step would receive only the label and the customer's words would be lost.
- The classifier uses `with_structured_output()` (Lec-30, Example 9) with `Literal[...]`, so the label is always exactly one of the three words.
- **Step 2, branch:** **`RunnableBranch((condition, chain), (condition, chain), default_chain)`** checks the conditions in order and runs only the **first** chain whose condition is true; if none is true, it runs the default (here: the mixed reply).
- We use `.assign()` once more to add the reply, so the output shows the feedback, its label and the reply together.

In [22]:
# Step a: load the API keys, create the model, and build the classifier chain
from typing import Literal                                  # a field that may hold only some fixed values
from pydantic import BaseModel                              # describe the classifier's answer
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders (Lec-31)
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text
from langchain_core.runnables import RunnableBranch, RunnableLambda, RunnablePassthrough   # if-else, a function as a step, "pass the input through"

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ
model = init_chat_model(
    "groq:openai/gpt-oss-120b",                             # free, on Groq
    # "openai:gpt-4o-mini",                                 # paid, on OpenAI: uncomment this line and comment out the line above
    max_retries=5,                                          # wait and retry when the free rate limit is hit
)

class Sentiment(BaseModel):                                 # the classifier's answer
    sentiment: Literal["positive", "negative", "mixed"]

classify_prompt = PromptTemplate.from_template("Classify the sentiment of this customer feedback: {feedback}")
classifier_model = model.with_structured_output(
    Sentiment,                                              # the answer is a Sentiment object
    method="json_schema",                                   # ask Groq for JSON that matches Sentiment
)

def get_label(result):
    '''Sentiment object → just the word.'''
    return result.sentiment

classify = classify_prompt | classifier_model | RunnableLambda(get_label)

print(classify.invoke({"feedback": "Excellent service, my order arrived a day early!"}))

positive


In [23]:
# Step b: RunnablePassthrough.assign() keeps the input dict and ADDS a key with the classifier's result
add_sentiment = RunnablePassthrough.assign(sentiment=classify)

example = {"feedback": "The laptop arrived broken and nobody answers my emails."}
print(add_sentiment.invoke(example))                        # {feedback} → {feedback, sentiment}

{'feedback': 'The laptop arrived broken and nobody answers my emails.', 'sentiment': 'negative'}


In [24]:
# Step c: one reply chain per sentiment, and RunnableBranch to choose between them
positive_prompt = PromptTemplate.from_template(
    "Thank the customer warmly. Reply in two sentences to this customer feedback: {feedback}"
)
positive_chain = positive_prompt | model | StrOutputParser()

negative_prompt = PromptTemplate.from_template(
    "Apologise, and say the issue goes to the support team. Reply in two sentences to this customer feedback: {feedback}"
)
negative_chain = negative_prompt | model | StrOutputParser()

mixed_prompt = PromptTemplate.from_template(
    "Thank the customer for the good points and apologise for the problem. "
    "Reply in two sentences to this customer feedback: {feedback}"
)
mixed_chain = mixed_prompt | model | StrOutputParser()

def is_positive(data):
    return data["sentiment"] == "positive"

def is_negative(data):
    return data["sentiment"] == "negative"

branch = RunnableBranch(
    (is_positive, positive_chain),                          # (condition, chain): checked first
    (is_negative, negative_chain),                          # checked second
    mixed_chain,                                            # the default, when no condition is true
)

# A test with a hand-made sentiment (no classifier): "mixed" matches no condition, so the default runs
test = {"feedback": "The delivery was late but the product quality is excellent.", "sentiment": "mixed"}
print(branch.invoke(test))

Thank you for highlighting the excellent quality of our product; we truly appreciate your positive feedback. We apologize for the delayed delivery and are working to improve our shipping times.


In [25]:
# Step d: the whole pipeline: {feedback} → {feedback, sentiment} → {feedback, sentiment, reply}
add_reply = RunnablePassthrough.assign(reply=branch)
feedback_chain = add_sentiment | add_reply

feedbacks = [
    "Excellent service, my order arrived a day early!",
    "The laptop arrived broken and nobody answers my emails.",
    "The delivery was late but the product quality is excellent.",
]
inputs = []
for feedback in feedbacks:
    inputs.append({"feedback": feedback})

results = feedback_chain.batch(inputs)                      # all three at the same time
for result in results:
    print(f"Feedback:  {result['feedback']}")
    print(f"Sentiment: {result['sentiment']}")
    print(f"Reply:     {result['reply']}\n")

Feedback:  Excellent service, my order arrived a day early!
Sentiment: positive
Reply:     Thank you so much for your kind words—we’re thrilled you loved the service! We’re delighted your order arrived early and look forward to serving you again soon.

Feedback:  The laptop arrived broken and nobody answers my emails.
Sentiment: negative
Reply:     We’re very sorry to hear that your laptop arrived damaged and that you haven’t received a response to your emails. I’ve escalated this to our support team, and they’ll reach out to you promptly to resolve the issue.

Feedback:  The delivery was late but the product quality is excellent.
Sentiment: mixed
Reply:     Thank you for your kind words about the excellent product quality. We apologize for the delayed delivery and appreciate your patience.



# <span style='background :lightgreen' >5. Making Chains Robust</span>

<div style="text-align:center;">
    <img src="../images/langchain-new-18.png"
         style="max-width:1500px; width:100%; height:auto; display:inline-block;">
</div>

## <span style='color:#7e22ce'>Example 7: <code>with_retry()</code> and <code>with_fallbacks()</code></span>
### Step 1: `with_retry()`: try the same step again
- `with_retry(stop_after_attempt=3)` returns a new Runnable that, on an error, waits a little and tries **again**, up to 3 times in all.
- To see it without spending any tokens, we use a small function that fails on its first two calls, as a busy server would.

In [26]:
# Step 1a: a step that fails on its first two calls, like a server that is busy for a moment (no model, no tokens)
from langchain_core.runnables import RunnableLambda         # turns a Python function into a chain step

attempts = {"count": 0}                                     # counts the calls

def busy_server(question):
    '''Fails on the first two calls, like a server that is busy for a moment.'''
    attempts["count"] += 1
    print(f"   attempt {attempts['count']} …")
    if attempts["count"] < 3:
        raise ConnectionError("the server is busy")         # a temporary error
    return f"answer to '{question}'"

step = RunnableLambda(busy_server)

try:
    print(step.invoke("What is LCEL?"))                     # without retry: the first error stops us
except ConnectionError as error:
    print("Without retry:", type(error).__name__, "-", error)

   attempt 1 …
Without retry: ConnectionError - the server is busy


In [27]:
# Step 1b: with_retry(): on an error, wait a little and try the same step again
attempts["count"] = 0                                       # start counting again

retrying_step = step.with_retry(stop_after_attempt=3)       # up to 3 attempts in all
print(retrying_step.invoke("What is LCEL?"))

   attempt 1 …
   attempt 2 …
   attempt 3 …
answer to 'What is LCEL?'


### Step 2: `with_fallbacks()`: switch to a backup chain
- `main_chain.with_fallbacks([backup_chain])` runs `main_chain`; if it raises an error, it runs `backup_chain` with the **same** input instead.
- To force an error, the main chain asks Groq for a model that does not exist. The backup chain uses a **free OpenRouter model** (Lec-30, Example 4) with the same prompt.
- In a real app we would combine both: `main_chain.with_retry(stop_after_attempt=3).with_fallbacks([backup_chain])`.

In [28]:
# Step 2a: load the API keys, and build a main chain that fails: its model does not exist
from dotenv import load_dotenv                              # reads the API keys from a file
from langchain.chat_models import init_chat_model           # creates a chat model from "provider:model"
from langchain_core.prompts import PromptTemplate           # a text template with named placeholders
from langchain_core.output_parsers import StrOutputParser   # AIMessage → its text

load_dotenv("../keys/.env", override=True)                  # put the keys from the file into os.environ

greet_prompt = PromptTemplate.from_template("Greet {name} in Urdu, in one line.")

broken_model = init_chat_model(
    "groq:openai/gpt-oss-999b",                             # this model does not exist on Groq
    max_retries=0,                                          # fail at once, without retrying
)
main_chain = greet_prompt | broken_model | StrOutputParser()

name = {"name": "Arif"}
try:
    main_chain.invoke(name)                                 # alone, the main chain fails
except Exception as error:
    message = str(error)
    print("Main chain alone:", type(error).__name__, "-", message[:90], "…")

Main chain alone: NotFoundError - Error code: 404 - {'error': {'message': 'The model `openai/gpt-oss-999b` does not exist or …


In [29]:
# Step 2b: a backup chain: the same prompt, with a free model on OpenRouter (Lec-30, Example 4)
backup_model = init_chat_model(
    "openrouter:nvidia/nemotron-3-ultra-550b-a55b:free",    # free, on OpenRouter (needs OPENROUTER_API_KEY)
    max_tokens=2000,                                        # room for the model's reasoning and the answer
)
backup_chain = greet_prompt | backup_model | StrOutputParser()
print(type(backup_model).__name__)                          # init_chat_model made the ChatOpenRouter of Lec-30

ChatOpenRouter


In [30]:
# Step 2c: with_fallbacks(): if the main chain raises an error, run the backup with the same input
robust_chain = main_chain.with_fallbacks([backup_chain])
print("With a fallback:", robust_chain.invoke(name))

# In a real app we would combine both:
# main_chain.with_retry(stop_after_attempt=3).with_fallbacks([backup_chain])

With a fallback: السلام علیکم عارف، آپ کیسے ہیں؟


# <span style='background :lightgreen' >6. Where Chains Go Next</span>

| Where | What the chain does there | Lecture |
|---|---|---|
| **RAG** | `retriever` → `prompt` → `model` → `parser`: answers questions from your own documents, with `RunnableParallel` and `RunnablePassthrough` | Lec-38 |
| **Agents** | An agent is not a fixed chain: the model decides the next step in a loop. We write that loop ourselves, then use `create_agent()` | Lec-39 to 51 |
| **LangGraph** | Workflows with loops, saved state, branches between agents and human approval | A later lecture |

# <span style='background :lightgreen' >7. Summary</span>

| Without chains (Lec-31) | With chains (LCEL) |
|---|---|
| `prompt.invoke()`, then `model.invoke()`, then `.text`, passed along by our code | `prompt \| model \| StrOutputParser()`, one `invoke()` |
| A loop of calls for many inputs | `chain.batch([...])` |
| Asking for JSON and parsing it ourselves | `PydanticOutputParser` as the last step |
| Reshaping data between steps by hand | `RunnableLambda(function)` |
| Running steps one after the other | `RunnableParallel({...})` |
| An `if` around separate calls | `RunnablePassthrough.assign(...)` + `RunnableBranch(...)` |
| `try/except` and our own retry loop | `.with_retry()` and `.with_fallbacks([...])` |

### Key takeaways
- Prompts, models, parsers and functions are all **Runnables**: they share `invoke`, `batch` and `stream`, so they can be joined with `|`.
- A **chain** passes each step's output to the next step, and is itself a Runnable.
- **Output parsers** turn the model's answer into a `str` (`StrOutputParser`) or a validated object (`PydanticOutputParser`).
- Three shapes cover most pipelines: **sequential** (`|`, with `RunnableLambda` to reshape data), **parallel** (`RunnableParallel`), and **branching** (`RunnableBranch`, with `.assign()` to keep the input).
- `with_retry()` survives short failures and `with_fallbacks()` switches to a backup: both matter with free models.
- The old `LLMChain`-style classes are deprecated; loops and state belong to LangGraph.
- **Next lecture (Lec-34):** **Document loaders**: getting your own documents into LangChain.

### Check your understanding
1. What do prompt templates, models and output parsers have in common that lets them be joined with `|`?
2. In Example 1, which line of Step 1 does `StrOutputParser()` replace?
3. What does `translate_chain.batch(inputs)` return, and in which order?
4. What are the two halves of `PydanticOutputParser`'s work, before and after the model? What happens if the model's JSON is wrong?
5. Why could `translate_chain` and `summarize_chain` not be joined directly in Example 4? What did `RunnableLambda` do?
6. What does `RunnableParallel` return, and why does that fit `merge_prompt`? When does running in parallel save time, and when not?
7. In Example 6, what would the reply prompt receive without `RunnablePassthrough.assign()`?
8. How does `RunnableBranch` choose a branch, and what happens when no condition is true?
9. When would you use `with_retry()`, and when `with_fallbacks()`?
10. Why does LangChain recommend LangGraph instead of chains for workflows with loops?